# 跑模型

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
import random
import numpy as np
import os
from tqdm import tqdm
import zipfile
import faiss
from torch_geometric.loader import NeighborLoader, GraphSAINTNodeSampler
from torch_geometric.utils import negative_sampling
import time

# ==================== 固定随机种子 ====================
torch.backends.cudnn.deterministic = True
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# ==================== 配置 ====================
scheme_type = "s8_gSage_EHD_1"
BASE_DIR = "/Users/minkexiu/Downloads/GitHub/Tianchi_EcommerceKG_mac"
TRAIN_FILE_PATH = f"{BASE_DIR}/originalData/OpenBG500/OpenBG500_train.tsv"
DEV_FILE_PATH = f"{BASE_DIR}/originalData/OpenBG500/OpenBG500_dev.tsv"
TEST_FILE_PATH = f"{BASE_DIR}/originalData/OpenBG500/OpenBG500_test.tsv"
OUTPUT_FILE_PATH = f"{BASE_DIR}/preprocessedData/OpenBG500_test.tsv"

MODEL_DIR = f"{BASE_DIR}/trained_models/{scheme_type}"
os.makedirs(MODEL_DIR, exist_ok=True)

TRAINED_MODEL_PATHS = {
    'GraphSAGE': f"{MODEL_DIR}/graphsage.pth",
    'TransE': f"{MODEL_DIR}/transE.pth",
    'TransH': f"{MODEL_DIR}/transH.pth",
    'TransD': f"{MODEL_DIR}/transD.pth"
}

# 超参数
EMBEDDING_DIM = 100
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-5
EPOCHS = 1
BATCH_SIZE = 256
NEGATIVE_SAMPLES = 10
MAX_LINES = None
LR_DECAY_STEP = 5
LR_DECAY_FACTOR = 0.1
FORCE_RETRAIN = False

# ==================== 数据集 ====================
class KnowledgeGraphDataset(torch.utils.data.Dataset):
    def __init__(self, file_path, is_test=False, max_lines=None, is_train=False):
        self.triples = []
        self.is_train = is_train
        self._load_data(file_path, is_test, max_lines)
    def _load_data(self, file_path, is_test, max_lines):
        print(f"加载数据: {file_path}")
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            if max_lines:
                lines = lines[:max_lines]
            for line in lines:
                parts = line.strip().split()
                if len(parts) == 3:
                    h, r, t = parts
                    self.triples.append((h, r, t))
                elif is_test and len(parts) == 2:
                    h, r = parts
                    self.triples.append((h, r, "<UNK>"))
        print(f"共加载 {len(self.triples)} 个三元组")
    def __len__(self):
        return len(self.triples)
    def __getitem__(self, idx):
        return self.triples[idx]

def collate_fn(batch):
    h_list, r_list, t_list = zip(*batch)
    return list(h_list), list(r_list), list(t_list)

# ==================== 映射器 ====================
class EntityRelationMapper:
    def __init__(self):
        self.entity_to_id = {}
        self.id_to_entity = {}
        self.relation_to_id = {}
        self.id_to_relation = {}
        self.entity_count = 0
        self.relation_count = 0
        self.all_train_triples = []

    def build_mappings(self, *datasets):
        entities = set()
        relations = set()
        for dataset in datasets:
            for h, r, t in dataset.triples:
                entities.add(h)
                entities.add(t)
                relations.add(r)
                if dataset.is_train:
                    self.all_train_triples.append((h, r, t))
        for e in sorted(entities):
            self.entity_to_id[e] = self.entity_count
            self.id_to_entity[self.entity_count] = e
            self.entity_count += 1
        for r in sorted(relations):
            self.relation_to_id[r] = self.relation_count
            self.id_to_relation[self.relation_count] = r
            self.relation_count += 1

# ==================== GraphSAGE 模型 ====================
class GraphSAGE(nn.Module):
    def __init__(self, num_entities, dim):
        super().__init__()
        self.embedding = nn.Embedding(num_entities, dim)
        nn.init.xavier_uniform_(self.embedding.weight)
        self.conv1 = SAGEConv(dim, dim, aggr='mean')
        self.conv2 = SAGEConv(dim, dim, aggr='mean')
        self.dropout = nn.Dropout(0.3)

    def forward(self, x, edge_index):
        x = self.embedding(x)
        x = self.dropout(torch.relu(self.conv1(x, edge_index)))
        x = self.conv2(x, edge_index)
        return x

    def get_embedding_matrix(self, device):
        # ❌ 错误：不要用 with torch.no_grad()
        # ✅ 正确：只在推理时用 no_grad，在训练时直接 forward
        x = torch.arange(self.embedding.num_embeddings, device=device)
        return self.forward(x, self.edge_index)

# ==================== TransE ====================
class TransE(nn.Module):
    def __init__(self, num_entities, num_relations, dim):
        super().__init__()
        self.E = nn.Embedding(num_entities, dim)
        self.R = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.E.weight)
        nn.init.xavier_uniform_(self.R.weight)

    def forward(self, h, r, t):
        return torch.norm(self.E(h) + self.R(r) - self.E(t), p=1, dim=1)

    def get_query_embedding(self, h, r):
        return self.E(h) + self.R(r)

    def normalize_entities(self):
        with torch.no_grad():
            self.E.weight.data.div_(torch.norm(self.E.weight.data, dim=1, keepdim=True) + 1e-9)

# ==================== TransH ====================
class TransH(nn.Module):
    def __init__(self, num_entities, num_relations, dim):
        super().__init__()
        self.E = nn.Embedding(num_entities, dim)
        self.R = nn.Embedding(num_relations, dim)
        self.W = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.E.weight)
        nn.init.xavier_uniform_(self.R.weight)
        nn.init.xavier_uniform_(self.W.weight)

    def project(self, emb, w):
        norm_w = torch.nn.functional.normalize(w, p=2, dim=1)
        scale = torch.sum(emb * norm_w, dim=1, keepdim=True)
        return emb - scale * norm_w

    def forward(self, h, r, t):
        h_emb = self.E(h)
        t_emb = self.E(t)
        r_vec = self.R(r)
        W = self.W(r)
        h_proj = self.project(h_emb, W)
        t_proj = self.project(t_emb, W)
        return torch.norm(h_proj + r_vec - t_proj, p=1, dim=1)

    def get_query_embedding(self, h, r):
        h_emb = self.E(h)
        r_vec = self.R(r)
        W = self.W(r)
        h_proj = self.project(h_emb, W)
        return h_proj + r_vec

    def normalize_entities(self):
        with torch.no_grad():
            self.E.weight.data.div_(torch.norm(self.E.weight.data, dim=1, keepdim=True) + 1e-9)

# ==================== TransD ====================
class TransD(nn.Module):
    def __init__(self, num_entities, num_relations, dim):
        super().__init__()
        self.E = nn.Embedding(num_entities, dim)
        self.R = nn.Embedding(num_relations, dim)
        self.E_proj = nn.Embedding(num_entities, dim)
        self.R_proj = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.E.weight)
        nn.init.xavier_uniform_(self.R.weight)
        nn.init.xavier_uniform_(self.E_proj.weight)
        nn.init.xavier_uniform_(self.R_proj.weight)

    def project(self, e, r_proj, e_proj):
        return e + torch.sum(e * e_proj, dim=1, keepdim=True) * r_proj

    def forward(self, h, r, t):
        h_emb = self.E(h)
        t_emb = self.E(t)
        r_vec = self.R(r)
        h_proj = self.project(h_emb, self.R_proj(r), self.E_proj(h))
        t_proj = self.project(t_emb, self.R_proj(r), self.E_proj(t))
        return torch.norm(h_proj + r_vec - t_proj, p=1, dim=1)

    def get_query_embedding(self, h, r):
        h_emb = self.E(h)
        r_vec = self.R(r)
        h_proj = self.project(h_emb, self.R_proj(r), self.E_proj(h))
        return h_proj + r_vec

    def normalize_entities(self):
        with torch.no_grad():
            self.E.weight.data.div_(torch.norm(self.E.weight.data, dim=1, keepdim=True) + 1e-9)

# ==================== 自定义负采样（修复 numpy 兼容性问题）====================
def custom_negative_sampling(edge_index, num_nodes, num_neg_samples=None):
    pos_edges = edge_index.cpu().numpy()
    pos_set = set(map(tuple, pos_edges.T))
    num_neg = num_neg_samples if num_neg_samples else edge_index.size(1)
    neg_edges = []
    for _ in range(num_neg):
        while True:
            h = random.randint(0, num_nodes - 1)
            t = random.randint(0, num_nodes - 1)
            if (h, t) not in pos_set and h != t:
                neg_edges.append([h, t])
                break
    return torch.tensor(neg_edges).t().contiguous().to(edge_index.device)

# ==================== 训练函数 ====================
def train_model(model, model_name, train_dataset, mapper, device, pretrained_E=None, pretrained_R=None):
    if os.path.exists(TRAINED_MODEL_PATHS[model_name]) and not FORCE_RETRAIN:
        print(f"[{model_name}] 模型已存在，跳过训练")
        return
    print(f"[{model_name}] 开始训练...")

    if pretrained_E is not None and hasattr(model, 'E'):
        print(f"✅ 使用上游 E 初始化 {model_name}.E")
        with torch.no_grad():
            model.E.weight.data.copy_(pretrained_E)
    if pretrained_R is not None and hasattr(model, 'R'):
        print(f"✅ 使用上游 R 初始化 {model_name}.R")
        with torch.no_grad():
            model.R.weight.data.copy_(pretrained_R)

    loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=LR_DECAY_STEP, gamma=LR_DECAY_FACTOR)
    model.to(device)
    model.train()

    for epoch in range(EPOCHS):
        epoch_loss = 0
        progress = tqdm(loader, desc=f"{model_name} Epoch {epoch+1}")
        for h_list, r_list, t_list in progress:
            h = torch.tensor([mapper.entity_to_id[h] for h in h_list], device=device)
            r = torch.tensor([mapper.relation_to_id[r] for r in r_list], device=device)
            t = torch.tensor([mapper.entity_to_id[t] for t in t_list], device=device)

            neg_t = torch.randint(0, mapper.entity_count, (len(h), NEGATIVE_SAMPLES), device=device)
            pos_t_expanded = t.unsqueeze(1).expand(-1, NEGATIVE_SAMPLES)
            mask = (neg_t == pos_t_expanded)
            while mask.any():
                neg_t[mask] = torch.randint(0, mapper.entity_count, (mask.sum(),), device=device)
                mask = (neg_t == pos_t_expanded)

            pos_score = model(h, r, t)
            neg_score = model(
                h.unsqueeze(1).expand(-1, NEGATIVE_SAMPLES).reshape(-1),
                r.unsqueeze(1).expand(-1, NEGATIVE_SAMPLES).reshape(-1),
                neg_t.reshape(-1)
            ).reshape(-1, NEGATIVE_SAMPLES)

            loss = torch.mean(torch.relu(pos_score.unsqueeze(1) - neg_score + 1.0))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if hasattr(model, 'normalize_entities'):
                model.normalize_entities()

            epoch_loss += loss.item()
            progress.set_postfix(loss=loss.item())

        scheduler.step()
        print(f"[{model_name}] Epoch {epoch+1} Loss: {epoch_loss / len(loader):.4f}")

    torch.save({
        'model_state_dict': model.state_dict(),
        'entity_count': mapper.entity_count,
        'relation_count': mapper.relation_count,
        'embedding_dim': EMBEDDING_DIM,
        'entity_to_id': mapper.entity_to_id,
        'relation_to_id': mapper.relation_to_id,
    }, TRAINED_MODEL_PATHS[model_name])
    print(f"[{model_name}] 模型已保存")

# ==================== GraphSAGE 训练 ====================
def train_graphsage(triples, mapper, device):
    """
    使用 GraphSAINTNodeSampler 训练 GraphSAGE（全设备兼容：CUDA, MPS, CPU）
    """
    # 检查模型是否已存在
    if os.path.exists(TRAINED_MODEL_PATHS['GraphSAGE']) and not FORCE_RETRAIN:
        print("[GraphSAGE] 模型已存在，跳过训练")
        # 加载时先到 CPU，再移动到目标 device
        ckpt = torch.load(TRAINED_MODEL_PATHS['GraphSAGE'], map_location='cpu')
        # 将嵌入矩阵移动到目标设备并返回
        return ckpt['E.weight'].to(device)

    print(f"🚀 开始训练 GraphSAGE (GraphSAINT) on {device.upper()}...")

    # 构建边列表（双向）
    edge_list = []
    for h, r, t in triples:
        if h in mapper.entity_to_id and t in mapper.entity_to_id:
            h_id = mapper.entity_to_id[h]
            t_id = mapper.entity_to_id[t]
            edge_list.append([h_id, t_id])
            edge_list.append([t_id, h_id])  # 无向图

    if len(edge_list) == 0:
        raise ValueError("训练数据为空或实体未正确映射")

    # 创建 PyG Data 对象
    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    data = Data(edge_index=edge_index, num_nodes=mapper.entity_count)

    # 🔥 使用 GraphSAINTNodeSampler
    train_loader = GraphSAINTNodeSampler(
        data,
        batch_size=128,
        walk_length=2,
        num_steps=5,
        sample_coverage=50,
        num_workers=0,
        persistent_workers=False,
    )

    # ✅ 1. 模型初始化：在 CPU 上创建，然后移动到目标设备
    model = GraphSAGE(mapper.entity_count, EMBEDDING_DIM)
    model = model.to(device)  # ✅ 关键：模型移动到 device
    model.train()

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    # 训练循环
    for epoch in range(EPOCHS):
        epoch_loss = 0
        start_time = time.time()

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:2d}", leave=False)
        for batch in pbar:
            # ✅ 2. 数据移动到目标设备
            batch = batch.to(device)

            # 前向传播
            x = torch.arange(batch.num_nodes, device=device)  # 确保 x 在 device 上
            z = model(x, batch.edge_index)

            # 正样本得分
            pos_score = (z[batch.edge_index[0]] * z[batch.edge_index[1]]).sum(dim=1)

            # 负样本得分
            neg_edge_index = negative_sampling(
                batch.edge_index,
                num_neg_samples=batch.edge_index.size(1),
                num_nodes=batch.num_nodes
            ).to(device)  # ✅ 移动到 device

            neg_score = (z[neg_edge_index[0]] * z[neg_edge_index[1]]).sum(dim=1)

            # 损失函数
            loss = -torch.log(torch.sigmoid(pos_score) + 1e-8).mean() \
                   - torch.log(1 - torch.sigmoid(neg_score) + 1e-8).mean()

            # 反向传播
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            pbar.set_postfix({"Loss": f"{loss.item():.4f}"})

        avg_loss = epoch_loss / len(train_loader)
        elapsed = time.time() - start_time
        print(f"Epoch {epoch+1:2d} | Avg Loss: {avg_loss:.4f} | Time: {elapsed:.2f}s")

    # ✅ 3. 推理：使用 device 上的模型生成嵌入
    print("🔄 正在生成全图实体嵌入...")
    model.eval()
    with torch.no_grad():
        full_x = torch.arange(mapper.entity_count, device=device)
        full_edge_index = edge_index.to(device)  # 移动到 device
        final_z = model(full_x, full_edge_index).detach()  # 在 device 上计算

    # ✅ 4. 保存：先移动到 CPU 保存（通用兼容）
    final_z_cpu = final_z.cpu()
    model_cpu = model.cpu()  # 保存前移回 CPU

    torch.save({
        'model_state_dict': model_cpu.state_dict(),
        'E.weight': final_z_cpu,
        'entity_count': mapper.entity_count,
        'embedding_dim': EMBEDDING_DIM,
        'entity_to_id': mapper.entity_to_id,
    }, TRAINED_MODEL_PATHS['GraphSAGE'])
    print("[GraphSAGE] 模型已保存 ✅")

    # ✅ 5. 返回值：确保返回 device 上的张量
    return final_z

# ==================== 加载模型 ====================
def load_model(model_class, path, mapper, device):
    ckpt = torch.load(path, map_location=device)
    model = model_class(ckpt['entity_count'], ckpt['relation_count'], EMBEDDING_DIM)
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(device)
    model.eval()
    return model

# ==================== 评估函数 ====================
def evaluate_model(model, dataset, mapper, device, k_list=(1, 3, 10)):
    print(f"📊 评估 {model.__class__.__name__} ...")
    model.eval()
    hits_at = {k: 0.0 for k in k_list}
    mrr = 0.0
    count = 0
    entity_emb = model.E.weight.data.cpu().numpy()
    index = faiss.IndexFlatL2(entity_emb.shape[1])
    index.add(entity_emb)

    with torch.no_grad():
        for h, r, t in tqdm(dataset.triples, desc="Evaluating"):
            try:
                h_id = torch.tensor([mapper.entity_to_id[h]], device=device)
                r_id = torch.tensor([mapper.relation_to_id[r]], device=device)
                t_id = mapper.entity_to_id[t]
            except KeyError:
                continue
            query = model.get_query_embedding(h_id, r_id).cpu().numpy()
            _, indices = index.search(query, 1000)
            pred_ids = indices[0]

            filtered_tails = [tail for head, rel, tail in mapper.all_train_triples if head == h and rel == r and tail != t]
            filter_ids = [mapper.entity_to_id[tail] for tail in filtered_tails if tail in mapper.entity_to_id]
            for fid in filter_ids:
                if fid in pred_ids:
                    mask = pred_ids == fid
                    pred_ids = np.concatenate([pred_ids[~mask], pred_ids[mask]])

            rank = np.where(pred_ids == t_id)[0]
            final_rank = rank[0] + 1 if len(rank) > 0 else 10000
            for k in k_list:
                if final_rank <= k:
                    hits_at[k] += 1
            mrr += 1.0 / final_rank
            count += 1

    for k in hits_at:
        hits_at[k] /= count
    mrr /= count
    print(f"HITS@1: {hits_at[1]:.4f}, HITS@3: {hits_at[3]:.4f}, HITS@10: {hits_at[10]:.4f}, MRR: {mrr:.4f}")
    return hits_at, mrr

# ==================== RRF 融合预测 ====================
def predict_ensemble(models_with_weights, test_dataset, mapper, device, rrf_k=60):
    print("🔍 开始融合预测 (RRF) ...")
    results = []
    sample_model = next(iter(models_with_weights.values()))[0]
    entity_emb = sample_model.E.weight.data.cpu().numpy()
    index = faiss.IndexFlatL2(entity_emb.shape[1])
    index.add(entity_emb)

    with torch.no_grad():
        for h, r, _ in tqdm(test_dataset.triples, desc="Predict"):
            try:
                h_id = torch.tensor([mapper.entity_to_id[h]], device=device)
                r_id = torch.tensor([mapper.relation_to_id[r]], device=device)
            except KeyError:
                preds = [h] * 10
                results.append('\t'.join([h, r] + preds))
                continue

            rrf_scores = np.zeros(mapper.entity_count)
            for name, (model, weight) in models_with_weights.items():
                q = model.get_query_embedding(h_id, r_id).cpu().numpy()
                _, indices = index.search(q, 1000)
                for rank, idx in enumerate(indices[0]):
                    rrf_scores[idx] += weight / (rrf_k + rank + 1)

            ranked = np.argsort(rrf_scores)[::-1][:10]
            preds = [mapper.id_to_entity[i] for i in ranked]
            results.append('\t'.join([h, r] + preds))

    os.makedirs(os.path.dirname(OUTPUT_FILE_PATH), exist_ok=True)
    with open(OUTPUT_FILE_PATH, 'w', encoding='utf-8') as f:
        f.write('\n'.join(results) + '\n')

    zip_path = OUTPUT_FILE_PATH.replace(".tsv", "") + f"__{scheme_type}.zip"
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(OUTPUT_FILE_PATH, arcname=os.path.basename(OUTPUT_FILE_PATH))
    print(f"✅ 预测结果已保存: {zip_path}")

# ==================== 主函数 ====================
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"🚀 使用设备: {device}")

    train_data = KnowledgeGraphDataset(TRAIN_FILE_PATH, max_lines=MAX_LINES, is_train=True)
    dev_data = KnowledgeGraphDataset(DEV_FILE_PATH, is_test=False, is_train=False)
    test_data = KnowledgeGraphDataset(TEST_FILE_PATH, is_test=True, is_train=False)

    mapper = EntityRelationMapper()
    mapper.build_mappings(train_data, dev_data, test_data)
    print(f"实体数: {mapper.entity_count}, 关系数: {mapper.relation_count}")

    # Step 1: GraphSAGE
    graphsage_E = train_graphsage(train_data.triples, mapper, device)

    # Step 2: TransE
    transE_model = TransE(mapper.entity_count, mapper.relation_count, EMBEDDING_DIM)
    train_model(transE_model, 'TransE', train_data, mapper, device, pretrained_E=graphsage_E)

    # Step 3: TransH ← TransE.E & TransE.R
    transE_ckpt = torch.load(TRAINED_MODEL_PATHS['TransE'], map_location=device)
    transE_E = transE_ckpt['model_state_dict']['E.weight']
    transE_R = transE_ckpt['model_state_dict']['R.weight']
    transH_model = TransH(mapper.entity_count, mapper.relation_count, EMBEDDING_DIM)
    train_model(transH_model, 'TransH', train_data, mapper, device, pretrained_E=transE_E, pretrained_R=transE_R)

    # Step 4: TransD ← TransH.E & TransH.R
    transH_ckpt = torch.load(TRAINED_MODEL_PATHS['TransH'], map_location=device)
    transH_E = transH_ckpt['model_state_dict']['E.weight']
    transH_R = transH_ckpt['model_state_dict']['R.weight']
    transD_model = TransD(mapper.entity_count, mapper.relation_count, EMBEDDING_DIM)
    train_model(transD_model, 'TransD', train_data, mapper, device, pretrained_E=transH_E, pretrained_R=transH_R)

    # 评估
    transE_eval = load_model(TransE, TRAINED_MODEL_PATHS['TransE'], mapper, device)
    transH_eval = load_model(TransH, TRAINED_MODEL_PATHS['TransH'], mapper, device)
    transD_eval = load_model(TransD, TRAINED_MODEL_PATHS['TransD'], mapper, device)

    evaluate_model(transE_eval, dev_data, mapper, device)
    evaluate_model(transH_eval, dev_data, mapper, device)
    evaluate_model(transD_eval, dev_data, mapper, device)

    # 融合预测
    models_with_weight = {
        'TransE': (transE_eval, 1.0),
        'TransH': (transH_eval, 1.5),
        'TransD': (transD_eval, 1.0),
    }
    predict_ensemble(models_with_weight, test_data, mapper, device)

    print("🎉 四阶段训练完成：GraphSAGE → TransE → TransH → TransD")

if __name__ == "__main__":
    main()